# "Visit with us" – Wellness Package Purchase Classifier (v2)

Predict which customers will buy the new Wellness Tourism Package (`ProdTaken`) so marketing can focus outreach.

Modernised from the original notebook (`legacy/`). Key fixes:
- **No target leakage** – the old notebook imputed missing values with group means that used `ProdTaken`, *before* splitting. Imputation now happens inside the CV pipeline.
- Removed APIs (`pandas_profiling`, `plot_roc_curve`, `base_estimator`) replaced with current scikit-learn / pandas.
- Hard-coded `class_weight={0:.18,1:.82}` → `balanced`; the decision threshold is tuned on out-of-fold predictions.
- Models compared by cross-validated PR-AUC, scored once on a held-out set.

Uses `data/Tourism.xlsx` if present, otherwise synthetic data with the same schema (numbers then are illustrative only).

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

from tourism_classifier.data import clean, load_raw, make_synthetic, TARGET
from tourism_classifier.report import feature_importance
from tourism_classifier.train import run

path = Path('../data/Tourism.xlsx')
raw = load_raw(path) if path.exists() else make_synthetic()
print('source:', path if path.exists() else 'synthetic', raw.shape)

source: synthetic (4000, 20)


## Data overview

In [2]:
df = clean(raw)
print(f'{df.shape=}  positive rate={df[TARGET].mean():.1%}')
df.isna().mean().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('missing share')

df.shape=(4000, 19)  positive rate=10.2%


,missing share
MonthlyIncome,0.04850
DurationOfPitch,0.04550
Age,0.04450
NumberOfTrips,0.03275
NumberOfChildrenVisiting,0.01200
NumberOfFollowups,0.01125
TypeofContact,0.00525
PreferredPropertyStar,0.00325


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col in zip(axes, ['ProductPitched', 'MaritalStatus', 'Passport']):
    sns.barplot(data=df, x=col, y=TARGET, ax=ax, errorbar=None)
    ax.set_ylabel('purchase rate'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout()

## Model comparison (5-fold out-of-fold, threshold tuned for F1)

In [4]:
res = run(raw, cv=5, n_jobs=-1)
res.cv_table.round(3)

,model,roc_auc,pr_auc,precision,recall,f1,threshold
0,logreg,0.821,0.357,0.354,0.541,0.428,0.701
1,random_forest,0.802,0.305,0.289,0.626,0.395,0.240
2,stacking,0.801,0.304,0.322,0.482,0.386,0.701
3,hist_gb,0.788,0.293,0.274,0.616,0.379,0.504
4,xgboost,0.777,0.271,0.300,0.479,0.369,0.299


## Held-out performance of the best model

In [5]:
print(res.best_name, {k: round(v, 3) for k, v in res.test_metrics.items()})
X_te, y_te = res.X_test, res.y_test
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_estimator(res.model, X_te, y_te, ax=axes[0])
PrecisionRecallDisplay.from_estimator(res.model, X_te, y_te, ax=axes[1])
ConfusionMatrixDisplay.from_predictions(
    y_te, (res.model.predict_proba(X_te)[:, 1] >= res.threshold).astype(int), ax=axes[2])
plt.tight_layout()

logreg {'roc_auc': 0.797, 'pr_auc': 0.339, 'precision': 0.3, 'recall': 0.446, 'f1': 0.359, 'threshold': 0.701}


## What drives purchases? (permutation importance on hold-out)

In [6]:
feature_importance(res).head(10).round(4).to_frame('PR-AUC drop when shuffled')

,PR-AUC drop when shuffled
Passport,0.1224
NumberOfFollowups,0.0728
MonthlyIncome,0.0667
Age,0.0385
Designation,0.0299
MaritalStatus,0.0144
ProductPitched,0.0082
NumberOfPersonVisiting,0.0042
NumberOfTrips,0.0022
Gender,0.0022


## Takeaways
- Choose the threshold by campaign economics: raise `--beta` (e.g. 2) to favour recall when contacts are cheap, lower it when each contact is expensive.
- Passport holders, single/unmarried customers and follow-up intensity were the strongest signals in the original analysis; check importances on the real data before acting.
- Retrain with `tourism-train --data data/Tourism.xlsx`.